# Lab: Đánh giá mô hình phân loại

## 1. Vì sao chỉ Accuracy không đủ?

Accuracy = tỷ lệ dự đoán đúng. Nghe đơn giản, nhưng gây hiểu nhầm ở các bài toán **mất cân bằng**: nếu 95% dataset là lớp 0, một model luôn dự đoán "0" đạt 95% accuracy mà không học được gì.

Ví dụ y học: chẩn đoán ung thư trong 1000 người khoẻ + 10 người bệnh. Model luôn nói "khoẻ" đạt 99% accuracy nhưng bỏ sót toàn bộ bệnh nhân.

Vì vậy ta cần các chỉ số khác: **Precision, Recall, F1, ROC-AUC**.

## 2. Confusion Matrix: bảng nền tảng

Với phân loại nhị phân (positive = lớp "có" / negative = lớp "không"):

Bảng dưới đây viết theo đúng thứ tự mà `sklearn.metrics.confusion_matrix(y_true, y_pred)` trả về: hàng là nhãn thật, cột là dự đoán, lớp negative (0) đứng trước lớp positive (1).

| | Dự đoán Negative (0) | Dự đoán Positive (1) |
|---|---|---|
| **Thực Negative (0)** | TN (True Negative) | FP (False Positive) |
| **Thực Positive (1)** | FN (False Negative) | TP (True Positive) |

Tức là ma trận sklearn có dạng
$$\begin{pmatrix}TN & FP \\ FN & TP\end{pmatrix}$$
Nhiều tài liệu khác đặt TP ở góc trên trái, vậy nên cần xác định layout trước khi đọc.

## 3. Bốn chỉ số chính

### Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
Tỷ lệ đoán đúng tổng. Chỉ tin cậy khi class cân bằng.

### Precision (độ chính xác)
$$\text{Precision} = \frac{TP}{TP + FP}$$
Trong số các mẫu **được dự đoán** là positive, bao nhiêu thật sự là positive? Cao = ít báo động giả.

### Recall (độ thu hồi / sensitivity)
$$\text{Recall} = \frac{TP}{TP + FN}$$
Trong số các mẫu **thực sự** là positive, bao nhiêu được model bắt được? Cao = ít bỏ sót.

### F1-score
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
Trung bình điều hoà của Precision và Recall, phạt nặng khi *một trong hai* thấp.

### Specificity
$$\text{Specificity} = \frac{TN}{TN + FP}$$
Tỷ lệ negative thực sự được nhận ra đúng, bằng $1 - \text{FPR}$ (False Positive Rate).

### 3.1. Confusion matrix và bảng đầy đủ các chỉ số suy ra từ nó

![Giải phẫu confusion matrix](images/01_giai_phau_confusion_matrix.png)

*Bốn ô, bốn cách diễn đạt bằng lời. Cách đọc tên: chữ thứ hai là điều model nói, chữ thứ nhất cho biết model nói đúng hay sai. "False Negative" là model nói Negative và nói sai, tức một ca dương bị bỏ sót.*

Toàn bộ phần còn lại của bài học này chỉ là các cách khác nhau để gộp bốn con số TP/FP/TN/FN thành một con số duy nhất. Dưới đây là bảng đầy đủ:

| Chỉ số | Công thức | Đọc bằng lời | Tên gọi khác |
|---|---|---|---|
| **Accuracy** | $\dfrac{TP+TN}{TP+TN+FP+FN}$ | "Trong tất cả, tôi đoán đúng bao nhiêu phần?" | (không có) |
| **Precision** | $\dfrac{TP}{TP+FP}$ | "Trong những ca tôi kêu là dương, bao nhiêu đúng là dương?" | PPV (Positive Predictive Value) |
| **Recall** | $\dfrac{TP}{TP+FN}$ | "Trong những ca thật sự dương, tôi bắt được bao nhiêu?" | TPR, Sensitivity, Hit rate |
| **Specificity** | $\dfrac{TN}{TN+FP}$ | "Trong những ca thật sự âm, tôi bỏ qua đúng bao nhiêu?" | TNR, Selectivity |
| **FPR** | $\dfrac{FP}{FP+TN} = 1-\text{Spec}$ | "Trong những ca thật sự âm, tôi báo động giả bao nhiêu?" | Fall-out, sai lầm loại I |
| **FNR** | $\dfrac{FN}{FN+TP} = 1-\text{Recall}$ | "Trong những ca thật sự dương, tôi bỏ sót bao nhiêu?" | Miss rate, sai lầm loại II |
| **NPV** | $\dfrac{TN}{TN+FN}$ | "Khi tôi nói âm, bao nhiêu lần đúng là âm?" | Negative Predictive Value |
| **Balanced accuracy** | $\dfrac{\text{Recall} + \text{Specificity}}{2}$ | "Accuracy nếu hai lớp bằng nhau về số lượng" | Macro-average recall |
| **$F_\beta$** | $(1+\beta^2)\dfrac{P \cdot R}{\beta^2 P + R}$ | "Gộp precision và recall, $\beta$ quyết định ưu tiên bên nào" | $F_1$ khi $\beta=1$ |

Cách nhớ mẫu số (chỗ nhiều sinh viên nhầm):

- Mẫu số là một cột (những gì model nói): precision, NPV. Đứng từ phía dự đoán mà nhìn.
- Mẫu số là một hàng (sự thật): recall, specificity, FPR, FNR. Đứng từ phía sự thật mà nhìn.

Vì thế recall và specificity không đổi khi tỷ lệ lớp thay đổi (chúng chuẩn hoá theo hàng), còn precision thay đổi mạnh khi lớp dương trở nên hiếm. Đây là gốc rễ của các hiện tượng ở phần dữ liệu mất cân bằng phía sau.


### 3.2. $F_\beta$ tổng quát và vì sao dùng trung bình điều hoà

$$F_\beta = (1+\beta^2)\cdot\frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

| $\beta$ | Ý nghĩa | Dùng khi |
|---|---|---|
| $\beta = 0.5$ | Coi precision quan trọng gấp 2 recall | Lọc spam, gợi ý sản phẩm, vì báo động giả gây khó chịu |
| $\beta = 1$ | Cân bằng ($F_1$) | Mặc định khi chưa biết ưu tiên gì |
| $\beta = 2$ | Coi recall quan trọng gấp 2 precision | Sàng lọc ung thư, phát hiện cháy rừng, vì bỏ sót gây hậu quả nặng |

Cách nhớ: $\beta$ là *"recall đáng giá bao nhiêu lần precision"*.

Vì sao không dùng trung bình cộng? Xét một model chỉ kêu dương đúng một ca mà nó chắc chắn nhất, và kêu âm cho tất cả 999 ca còn lại.

- Precision $= 1/1 = 1.00$ (nó nói dương đúng một lần và trúng)
- Recall $= 1/50 = 0.02$ (thực tế có 50 ca dương, nó bắt được 1)

| Cách gộp | Kết quả | Nhận xét |
|---|---|---|
| Trung bình cộng $\dfrac{1.00+0.02}{2}$ | 0.510 | nghe như "hơn 50%, tạm được", nhưng model này gần như vô dụng |
| Trung bình điều hoà $2\dfrac{1.00 \times 0.02}{1.00+0.02}$ | 0.039 | gần bằng 0, phản ánh đúng chất lượng |

Trung bình điều hoà luôn nghiêng về phía số nhỏ hơn: nó chỉ cao khi cả hai đều cao. Về mặt toán, với $a,b>0$ ta luôn có $\text{HM} \le \text{GM} \le \text{AM}$, và HM bị kéo về 0 ngay khi một trong hai số tiến về 0. Đó chính xác là hành vi ta muốn ở một chỉ số tổng hợp precision và recall.


### 3.3. MCC và Cohen's Kappa: hai chỉ số dùng cả bốn ô

Cả accuracy lẫn F1 đều có điểm mù. F1 hoàn toàn bỏ qua TN (nhìn lại công thức: chỉ có TP, FP, FN). Hai chỉ số sau dùng cả bốn ô.

#### Matthews Correlation Coefficient (MCC)

$$\text{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

- Giá trị trong $[-1, +1]$: $+1$ = hoàn hảo, $0$ = đoán mò, $-1$ = sai hoàn toàn (đoán ngược).
- Bản chất là hệ số tương quan Pearson giữa vector nhãn thật và vector dự đoán (khi mã hoá $0/1$), nên nó có ý nghĩa thống kê rõ ràng.
- Chỉ cao khi model làm tốt cả bốn ô, nên không thể đạt điểm cao bằng cách đoán lệch về lớp đa số.
- Vì thế MCC được xem là chỉ số cân bằng nhất cho bài nhị phân mất cân bằng. Nó còn đối xứng: đổi vai trò lớp dương và lớp âm cho nhau, MCC không đổi, trong khi F1 thì đổi.

#### Cohen's Kappa

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

với $p_o$ = accuracy quan sát được, $p_e$ = accuracy kỳ vọng nếu đoán ngẫu nhiên nhưng vẫn giữ đúng tỷ lệ dự đoán của model:

$$p_e = \frac{(TP+FP)(TP+FN) + (TN+FN)(TN+FP)}{(TP+TN+FP+FN)^2}$$

Diễn giải: kappa cho biết model tốt hơn một bộ đoán ngẫu nhiên (giữ đúng tỷ lệ dự đoán ấy) được bao nhiêu phần trăm của khoảng cách còn lại tới mức hoàn hảo.

| $\kappa$ | Diễn giải thông dụng |
|---|---|
| < 0 | Kém hơn đoán ngẫu nhiên |
| 0.00 đến 0.20 | Rất yếu |
| 0.21 đến 0.40 | Yếu |
| 0.41 đến 0.60 | Trung bình |
| 0.61 đến 0.80 | Tốt |
| 0.81 đến 1.00 | Rất tốt |

```python
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score
matthews_corrcoef(y_true, y_pred)     # MCC
cohen_kappa_score(y_true, y_pred)     # Kappa
```

Trong thực hành, với bài nhị phân mất cân bằng nên báo cáo MCC cùng PR-AUC (Average Precision) thay vì accuracy cùng ROC-AUC. Kappa còn được dùng nhiều để đo mức đồng thuận giữa hai người gán nhãn (inter-annotator agreement), với cùng một công thức.


## 4. Precision-Recall trade-off

Hai chỉ số thường *đối kháng* nhau. Tăng ngưỡng quyết định:
- Model khắt khe hơn, ít dự đoán positive, nên precision tăng và recall giảm.
- Model dễ dãi hơn, dự đoán nhiều positive, nên recall tăng và precision giảm.

Tuỳ bài toán mà ưu tiên cái nào:
- Lọc spam: ưu tiên precision (tránh đẩy mail thật vào spam).
- Chẩn đoán ung thư: ưu tiên recall (tránh bỏ sót bệnh).
- Cần cân bằng: dùng F1.

## 5. ROC và AUC

ROC = Receiver Operating Characteristic. Vẽ TPR (= Recall) theo FPR (= 1 − Specificity) khi quét ngưỡng.

**AUC** (Area Under ROC) = diện tích dưới đường, $\in [0, 1]$:
- AUC = 1: model hoàn hảo.
- AUC = 0.5: đoán mò.
- AUC < 0.5: kém hơn đoán mò, nhưng có thể đảo ngược dự đoán để được > 0.5.

## 6. Multiclass: averaging methods

Khi có $> 2$ lớp, Precision/Recall/F1 tính riêng cho từng lớp rồi gộp lại bằng một trong ba cách:

- **Macro**: trung bình cộng đơn giản. Mỗi lớp có cùng trọng số.
- **Weighted**: trung bình có trọng số theo số mẫu mỗi lớp.
- **Micro**: cộng tất cả TP, FP, FN trên mọi lớp rồi tính một lần. Bằng accuracy.

Khi lớp mất cân bằng và muốn quan tâm đều các lớp thì dùng macro; khi quan tâm tổng thể thì dùng weighted.

---

## 7. Hình neo của cả bài: hai phân phối và một ngưỡng

Nếu chỉ nhớ một hình trong bài này, hãy nhớ hình dưới đây.

![Hai phân phối điểm số và ngưỡng](images/02_hai_phan_phoi_va_nguong.png)

*Model không trả về nhãn, nó trả về **điểm số**. Lớp âm (xanh) thường bị chấm điểm thấp, lớp dương (đỏ) thường được chấm điểm cao, nhưng hai phân phối chồng lấn. Ta chọn một ngưỡng để cắt. Ngưỡng cắt sinh ra bốn vùng TP/FP/TN/FN, và bốn vùng đó sinh ra mọi chỉ số.*

Hãy đọc hình theo đúng thứ tự này:

1. Vùng chồng lấn quyết định độ khó. Nếu hai phân phối tách rời hoàn toàn, có một ngưỡng cho accuracy = 100% và không cần học thêm gì nữa. Vùng chồng lấn càng lớn thì bài toán càng khó và mọi chỉ số càng thấp.
2. Ngưỡng không phải thuộc tính của model. Đổi ngưỡng không train lại gì cả, chỉ đọc lại cùng một điểm số theo một mốc khác. Vì thế precision, recall, F1, accuracy đều phụ thuộc ngưỡng, còn ROC-AUC và Average Precision thì không (chúng quét toàn bộ ngưỡng).
3. Kéo ngưỡng sang trái: vùng đỏ bên phải to ra, TP tăng và FN giảm (recall tăng), nhưng vùng cam FP cũng to ra (precision giảm).
4. Kéo ngưỡng sang phải: ngược lại. Không có cách nào được cả hai.

Vì vậy câu hỏi "model này chính xác bao nhiêu phần trăm?" là câu hỏi thiếu dữ kiện; phải hỏi thêm "ở ngưỡng nào?".


## 8. ROC sinh ra từ đâu và ý nghĩa xác suất của AUC

![Từ ngưỡng đến ROC](images/03_tu_nguong_den_roc.png)

*Trái: ba ngưỡng A, B, C trên chính hình neo ở mục 7. Phải: mỗi ngưỡng cho một cặp $(FPR, TPR)$, tức một chấm đỏ trên đồ thị. Trượt ngưỡng từ $+\infty$ về $-\infty$, chấm đỏ vẽ nên đường ROC. ROC chỉ là quỹ đạo của chấm đó.*

Đọc vị trí trên ROC:

| Vị trí | Ngưỡng tương ứng | Ý nghĩa |
|---|---|---|
| Góc dưới-trái $(0,0)$ | Ngưỡng $= +\infty$ | Không bao giờ kêu dương: không FP nào, nhưng cũng chẳng bắt được ai |
| Góc trên-phải $(1,1)$ | Ngưỡng $= -\infty$ | Kêu dương với tất cả: bắt hết, nhưng báo động giả toàn tập |
| Góc trên-trái $(0,1)$ | Có một ngưỡng đạt tới đây | Model hoàn hảo |
| Đường chéo | Ngưỡng nào cũng rơi vào đây | Đoán mò |

### AUC thật sự đo cái gì?

$$\boxed{\;\text{AUC} = P\big(\,\text{score}(x^+) > \text{score}(x^-)\,\big) + \tfrac{1}{2}P\big(\,\text{score}(x^+) = \text{score}(x^-)\,\big)\;}$$

Diễn giải: bốc ngẫu nhiên một mẫu dương và một mẫu âm, AUC là xác suất model chấm cho mẫu dương điểm cao hơn mẫu âm.

- AUC = 0.95 nghĩa là: 95 trên 100 cặp (dương, âm) được xếp đúng thứ tự.
- AUC = 0.5 nghĩa là model xếp thứ tự ngẫu nhiên.
- AUC không nói gì về việc model có đặt ngưỡng đúng chỗ không, cũng không nói gì về chất lượng xác suất. Một model có AUC 0.99 vẫn có thể trả về xác suất lệch xa thực tế (xem mục 12).

Liên hệ với thống kê Mann-Whitney U. Với $n^+$ mẫu dương và $n^-$ mẫu âm:

$$\text{AUC} = \frac{U}{n^+ \cdot n^-}$$

trong đó $U$ là thống kê Mann-Whitney U (còn gọi Wilcoxon rank-sum). Nghĩa là AUC chính là một thống kê hạng đã chuẩn hoá: nó chỉ quan tâm thứ tự của các điểm số, không quan tâm giá trị tuyệt đối. Hệ quả hữu ích: nhân đôi mọi điểm số, hay biến đổi chúng qua bất kỳ hàm tăng nghiêm ngặt nào (sigmoid, log...), AUC không đổi.


### Ảnh động: ngưỡng trượt và chấm chạy trên đường ROC

![Ảnh động cho thấy ngưỡng trượt sinh ra từng điểm trên đường ROC](images/anim_nguong_va_roc.gif)

*Bên trái, đường ngưỡng trượt dần từ 0.95 xuống 0.05 trên đúng hai phân phối điểm số của mục 7. Bên phải, chấm đỏ là cặp (FPR, TPR) ứng với chính ngưỡng đó, và nó di chuyển dọc theo đường ROC. Ngưỡng cao thì cả FPR lẫn TPR đều gần 0, chấm nằm ở góc dưới bên trái; ngưỡng thấp thì cả hai tiến về 1, chấm chạy lên góc trên bên phải. Nhìn bảng số liệu sẽ thấy recall tăng dần còn precision giảm dần khi ngưỡng đi xuống: đường ROC chỉ là dấu vết mà cái chấm này để lại.*


## 9. Chọn ngưỡng: theo F1 lớn nhất hay theo ma trận chi phí?

![Precision, Recall, F1 theo ngưỡng](images/04_precision_recall_theo_nguong.png)

*Ngưỡng 0.5 chỉ là mặc định của thư viện. Trên dữ liệu hơi mất cân bằng này, ngưỡng cho F1 lớn nhất là 0.59; dùng 0.5 làm mất 9 điểm F1.*

Có hai cách chọn ngưỡng; trong thực tế cách thứ hai thường hợp lý hơn.

### Cách 1: tối đa hoá một chỉ số (F1, $F_\beta$, Youden's J...)

```python
from sklearn.metrics import precision_recall_curve
p, r, thr = precision_recall_curve(y_val, y_score)
f1 = 2 * p * r / (p + r + 1e-12)
best_threshold = thr[np.argmax(f1[:-1])]
```

Lưu ý rằng phải chọn ngưỡng trên tập validation, không phải tập test. Chọn ngưỡng trên test rồi báo cáo kết quả trên chính test làm kết quả bị thiên lệch lạc quan (xem mục 13).

### Cách 2 (thường hợp lý hơn): tối thiểu hoá chi phí kỳ vọng

Trong thực tế, FP và FN không tốn kém như nhau. Gọi $C_{FP}$ là thiệt hại của một báo động giả, $C_{FN}$ là thiệt hại của một ca bỏ sót, và $p = P(y=1 \mid x)$ là xác suất model đưa ra. Khi đó:

$$\text{chi phí nếu đoán dương} = (1-p)\,C_{FP}, \qquad \text{chi phí nếu đoán âm} = p\,C_{FN}$$

Nên đoán dương khi $p\,C_{FN} > (1-p)\,C_{FP}$, tức là:

$$\boxed{\;t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}\;}$$

Ví dụ y tế. Bỏ sót một ca ung thư $C_{FN} = 100$ (đơn vị thiệt hại), báo động giả chỉ tốn thêm một lần xét nghiệm $C_{FP} = 1$:

$$t^* = \frac{1}{1 + 100} \approx 0.0099$$

Nghĩa là chỉ cần model nghi ngờ 1%, ta đã nên gọi bệnh nhân quay lại xét nghiệm. Dùng ngưỡng 0.5 ở đây là sai về mặt y khoa.

Ví dụ ngược, lọc spam. Đẩy nhầm một email quan trọng vào thùng rác rất tốn kém ($C_{FP} = 50$), để lọt một spam chỉ hơi khó chịu ($C_{FN} = 1$), nên $t^* = 50/51 \approx 0.98$: phải rất chắc chắn mới gắn nhãn spam.

Công thức này chỉ đúng khi $p$ là xác suất đã hiệu chỉnh. Nếu model trả về điểm số chưa hiệu chỉnh, hãy hiệu chỉnh trước (mục 12) rồi mới áp ngưỡng chi phí.


## 10. ROC hay Precision-Recall? Câu trả lời phụ thuộc mức mất cân bằng

![ROC vs PR khi mất cân bằng](images/05_roc_vs_pr_khi_mat_can_bang.png)

*Cùng một chất lượng model (hai phân phối điểm số giống hệt nhau), chỉ đổi tỷ lệ lớp từ 50/50 sang 1/99. ROC-AUC gần như không đổi (0.948 so với 0.953), trong khi Average Precision giảm từ 0.947 xuống 0.423. Đường PR phản ánh đúng độ khó thực tế hơn.*

Vì sao ROC gần như không đổi? Nhìn lại hai mẫu số:

$$FPR = \frac{FP}{FP+TN}, \qquad \text{Precision} = \frac{TP}{TP+FP}$$

Khi lớp âm chiếm 99%, $TN$ là một con số rất lớn. Thêm 500 FP vào mẫu số $FP+TN \approx 9900$ hầu như không nhúc nhích FPR. Nhưng chính 500 FP đó rơi thẳng vào mẫu số của precision (chỉ có ~100 mẫu dương để đối trọng) và kéo nó xuống mạnh.

| | ROC / ROC-AUC | Precision-Recall / AP |
|---|---|---|
| Trục | FPR vs TPR | Recall vs Precision |
| Có dùng TN không? | Có (trong FPR) | Không, chỉ nhìn TP, FP, FN |
| Bất biến với tỷ lệ lớp? | Có (vừa là ưu điểm vừa là điểm cần lưu ý) | Không, nó phản ánh đúng độ khó thực tế |
| Baseline (model đoán mò) | 0.5 (đường chéo) | Bằng tỷ lệ lớp dương (đường ngang) |
| Nên dùng khi | Hai lớp tương đối cân bằng; quan tâm cả hai lớp như nhau | Mất cân bằng; chỉ quan tâm lớp hiếm (gian lận, bệnh, lỗi) |

Khi đọc đường PR, lưu ý rằng AP = 0.42 nghe có vẻ thấp, nhưng baseline chỉ là 0.01, nghĩa là model tốt hơn ngẫu nhiên 42 lần. Nên so AP với tỷ lệ lớp dương, không so với 0.5.

### Average Precision và AUC-PR khác nhau chỗ nào?

$$\text{AP} = \sum_n (R_n - R_{n-1})\,P_n$$

- AP là trung bình precision có trọng số theo mức tăng của recall, một tổng rời rạc, không nội suy.
- AUC-PR là diện tích dưới đường PR, thường tính bằng quy tắc hình thang, tức nội suy tuyến tính giữa các điểm; đường PR có thể nhảy bậc mạnh nên AUC-PR hay lạc quan quá mức.
- Vì thế sklearn khuyến nghị dùng `average_precision_score` chứ không dùng `auc(recall, precision)`.

```python
from sklearn.metrics import average_precision_score, precision_recall_curve
ap = average_precision_score(y_test, y_score)     # nên dùng cái này
```


## 11. Macro / Micro / Weighted: ba con số cho cùng một model

![Macro micro weighted](images/07_macro_micro_weighted.png)

*Bài 3 lớp mất cân bằng (900 / 100 / 30 mẫu). Model làm rất tốt lớp A (F1 = 0.97) nhưng kém ở lớp C (F1 = 0.39). Ba cách gộp cho ra 0.659, 0.916 và 0.919, chênh nhau tới 26 điểm phần trăm trên cùng một bảng dự đoán.*

| Cách gộp | Công thức | Mỗi lớp nặng bao nhiêu | Nói lên điều gì |
|---|---|---|---|
| **Macro** | $\dfrac{1}{K}\sum_k F1_k$ | Bằng nhau, bất kể lớp to hay nhỏ | Model đối xử với mọi lớp ra sao; lớp hiếm làm kém là tụt ngay |
| **Weighted** | $\sum_k \dfrac{n_k}{n} F1_k$ | Tỷ lệ số mẫu | Trung bình một mẫu ngẫu nhiên được phục vụ ra sao; lớp lớn át hết |
| **Micro** | Gộp toàn bộ $TP, FP, FN$ rồi tính một lần | Tỷ lệ số mẫu | Với phân loại đơn nhãn, micro-F1 = micro-P = micro-R = accuracy |

Chọn thế nào:

- Mỗi lớp đều quan trọng như nhau (chẩn đoán bệnh hiếm, nhận dạng chữ viết tay): dùng macro.
- Quan tâm trải nghiệm trung bình của người dùng, lớp lớn quan trọng hơn thật: dùng weighted.
- Bài đa nhãn (multi-label) hoặc muốn một con số tổng thể: dùng micro.

Câu "model đạt F1 = 0.92" là chưa đủ thông tin với bài đa lớp nếu không nói rõ average nào. Trong ví dụ trên, cùng một model có thể được báo cáo là 0.92 (weighted) hoặc 0.66 (macro). Nên ghi rõ "macro-F1 = 0.659, weighted-F1 = 0.916", và kèm `classification_report` để người đọc nhìn thấy từng lớp, vì con số gộp che mất chính chỗ model đang yếu.


## 12. Hiệu chỉnh xác suất (Calibration): khi 0.9 phải thật sự nghĩa là 90%

![Calibration curve](images/08_calibration_curve.png)

*Trái: reliability diagram. Trục hoành = xác suất model nói; trục tung = tần suất dương thực tế trong nhóm đó. Model hiệu chỉnh tốt bám sát đường chéo. Phải: model quá tự tin dồn xác suất về hai đầu 0/1, model thiếu tự tin co cụm quanh 0.5.*

### Vì sao cần?

Nhãn đúng và xác suất đúng là hai chuyện khác nhau. Hai model có thể có cùng AUC 0.95 (xếp thứ tự y hệt nhau) mà một cái nói "90%" khi thực tế chỉ 72%. Với bài toán chỉ cần nhãn thì không sao. Nhưng khi xác suất được đưa vào một công thức thì sai lệch đó gây thiệt hại thật:

- Định giá bảo hiểm / tính vốn dự phòng rủi ro tín dụng: phí = xác suất vỡ nợ × thiệt hại.
- Chọn ngưỡng theo ma trận chi phí (mục 9): công thức $t^* = C_{FP}/(C_{FP}+C_{FN})$ chỉ đúng nếu $p$ đã hiệu chỉnh.
- Ghép nhiều model lại (ensemble bằng trung bình xác suất).
- Chuyển tiếp cho con người quyết định ("bác sĩ ơi, ca này 85% ác tính").

### Model nào hay bị lệch?

| Model | Xu hướng |
|---|---|
| Logistic Regression | Thường đã hiệu chỉnh khá tốt (vì nó tối thiểu trực tiếp log loss) |
| Naive Bayes | Thường quá tự tin: giả định độc lập bị vi phạm nên xác suất dồn về 0/1 |
| SVM (`decision_function`) | Không phải xác suất; cần Platt scaling |
| Random Forest | Thiếu tự tin ở hai đầu, trung bình của nhiều cây hiếm khi cho 0.0 hay 1.0 |
| Mạng nơ-ron sâu hiện đại | Thường quá tự tin |

### Hai cách hiệu chỉnh

| | **Platt scaling** (`method='sigmoid'`) | **Isotonic regression** (`method='isotonic'`) |
|---|---|---|
| Dạng | $P = \dfrac{1}{1+e^{A f(x)+B}}$, chỉ 2 tham số | Hàm bậc thang đơn điệu không giảm, phi tham số |
| Số mẫu cần | Ít (vài trăm là đủ) | Nhiều (từ khoảng 1000 trở lên) |
| Linh hoạt | Thấp, chỉ sửa được lệch dạng sigmoid | Cao, sửa được mọi kiểu lệch đơn điệu |
| Rủi ro | Không sửa nổi lệch phức tạp | Dễ overfit khi ít dữ liệu |

```python
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
cal = CalibratedClassifierCV(SVC(), method='sigmoid', cv=5).fit(X_train, y_train)
```

### Đo chất lượng xác suất: Brier score

$$\text{Brier} = \frac{1}{n}\sum_{i=1}^{n}\big(p_i - y_i\big)^2$$

Đây chính là MSE áp lên xác suất: càng nhỏ càng tốt, $0$ là hoàn hảo, $0.25$ là mức của model luôn nói "0.5". Brier là một **proper scoring rule**: nó chỉ đạt cực tiểu khi model báo cáo đúng xác suất thật, nên không thể đạt điểm tốt hơn bằng cách nói quá hoặc nói dè dặt.

Lưu ý rằng Brier gộp cả độ chính xác của xếp hạng lẫn độ chuẩn của hiệu chỉnh, nên cần báo cáo kèm reliability diagram: Brier thấp mà đường cong vẫn lệch là chuyện có thật.


# THỰC HÀNH 1: Tính từng chỉ số bằng tay rồi so với sklearn

Cho `y_true` và `y_pred` thật, ta tự tính TP/FP/TN/FN, rồi từ đó suy ra mọi chỉ số. Cuối cùng so với sklearn để đối chiếu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, accuracy_score,
                              precision_score, recall_score, f1_score,
                              classification_report, roc_curve, auc)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer, load_iris

np.random.seed(42)

# Ví dụ tay: 10 mẫu, 1 = positive, 0 = negative
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 1])

TP = int(((y_true == 1) & (y_pred == 1)).sum())
TN = int(((y_true == 0) & (y_pred == 0)).sum())
FP = int(((y_true == 0) & (y_pred == 1)).sum())
FN = int(((y_true == 1) & (y_pred == 0)).sum())
print(f'TP = {TP}, TN = {TN}, FP = {FP}, FN = {FN}')

acc       = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
spec      = TN / (TN + FP) if (TN + FP) > 0 else 0

print(f'\nTay:    acc={acc:.3f}  prec={precision:.3f}  rec={recall:.3f}  f1={f1:.3f}  spec={spec:.3f}')
print(f'sklearn:acc={accuracy_score(y_true, y_pred):.3f}  '
      f'prec={precision_score(y_true, y_pred):.3f}  '
      f'rec={recall_score(y_true, y_pred):.3f}  '
      f'f1={f1_score(y_true, y_pred):.3f}')

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix sklearn (hàng = nhãn thật, cột = dự đoán):')
print(cm)
print(f'\nTN = cm[0,0] = {cm[0,0]}')
print(f'FP = cm[0,1] = {cm[0,1]}')
print(f'FN = cm[1,0] = {cm[1,0]}')
print(f'TP = cm[1,1] = {cm[1,1]}')

# THỰC HÀNH 2: Vì sao accuracy gây hiểu nhầm khi mất cân bằng

Sinh dữ liệu mất cân bằng 95/5, dựng một baseline luôn dự đoán lớp đa số, xem các chỉ số lệch tới đâu. Accuracy in ra rất cao dù baseline không học gì; F1 bằng 0 vì recall bằng 0.

In [ ]:
# 950 negative, 50 positive
y_imb = np.array([0] * 950 + [1] * 50)
y_dummy = np.zeros_like(y_imb)   # luôn dự đoán 0

print(f'Accuracy:  {accuracy_score(y_imb, y_dummy)*100:.2f}%')
print(f'Precision: {precision_score(y_imb, y_dummy, zero_division=0)*100:.2f}%')
print(f'Recall:    {recall_score(y_imb, y_dummy)*100:.2f}%')
print(f'F1:        {f1_score(y_imb, y_dummy)*100:.2f}%')

### Nhìn bằng hình: accuracy gây hiểu nhầm đến mức nào

![Năm chỉ số trên dữ liệu 95/5](images/06_accuracy_lua_doi.png)

*Trên dữ liệu 95/5, baseline luôn đoán lớp đa số có accuracy cao hơn mô hình thật sự có học (0.950 so với 0.943), dù nó bỏ sót toàn bộ ca hiếm. Chỉ khi nhìn F1, Kappa, MCC (đều bằng 0) và balanced accuracy (bằng 0.5, ngang mức ngẫu nhiên) thì sự khác biệt mới hiện ra.*

Bốn kết luận rút ra từ biểu đồ này:

1. Accuracy cao hơn không có nghĩa là model tốt hơn. Đây không phải trường hợp hiếm; nó xảy ra thường xuyên khi lớp hiếm dưới khoảng 10%.
2. F1 của lớp hiếm bằng 0 vì recall bằng 0.
3. Balanced accuracy bằng 0.5 là mốc ngẫu nhiên dễ thấy nhất; nó bằng trung bình recall của hai lớp, $(1.0 + 0.0)/2$.
4. MCC bằng 0 vì tử số $TP \cdot TN - FP \cdot FN = 0 \cdot 950 - 0 \cdot 50 = 0$: dự đoán không có tương quan với nhãn thật.

Trong thực hành, nên chạy baseline `DummyClassifier(strategy='most_frequent')` trước khi báo cáo kết quả. Nếu model không vượt nó rõ ràng trên F1 hoặc MCC (chứ không phải accuracy), model chưa học được gì đáng kể.


# THỰC HÀNH 3: Đánh giá KNN trên Breast Cancer

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Scale rồi train KNN
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)
y_proba = knn.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred, target_names=data.target_names))

In [ ]:
# vẽ confusion matrix dạng heatmap, ghi số vào từng ô
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(data.target_names); ax.set_yticklabels(data.target_names)
ax.set_xlabel('Dự đoán'); ax.set_ylabel('Thật')
ax.set_title('Confusion matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# ROC + AUC
fpr, tpr, thresh = roc_curve(y_test, y_proba)
auc_val = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, linewidth=2, label=f'KNN (AUC = {auc_val:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Đoán mò (AUC = 0.5)')
plt.xlabel('FPR (1 − Specificity)'); plt.ylabel('TPR (Recall)')
plt.title('ROC curve: Breast Cancer')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

# THỰC HÀNH 4: Multiclass với Iris

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(scaler.fit_transform(X_train), y_train)
y_pred = knn.predict(scaler.transform(X_test))

# 3 cách average khác nhau
print('Classification report (mặc định macro & weighted):')
print(classification_report(y_test, y_pred, target_names=iris.target_names))
print(f'F1 macro:    {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'F1 weighted: {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'F1 micro:    {f1_score(y_test, y_pred, average="micro"):.4f}   (= accuracy)')

## 13. Cross-validation: một lần chia dữ liệu là chưa đủ

![k-fold cross validation](images/09_kfold_cross_validation.png)

*Một lần `train_test_split` chỉ cho một con số, và con số đó phụ thuộc vào `random_state` may hay rủi. k-fold cho ra $k$ con số, nên ta có cả trung bình lẫn độ dao động.*

| Kiểu | Cách chia | Dùng khi |
|---|---|---|
| **K-Fold** | Chia $k$ phần bằng nhau, lần lượt mỗi phần làm validation | Dữ liệu cân bằng, không có cấu trúc đặc biệt |
| **Stratified K-Fold** | Như trên nhưng giữ nguyên tỷ lệ lớp trong từng fold | Mặc định nên dùng cho phân loại; sklearn tự chọn khi `cv=5` với classifier |
| **Leave-One-Out (LOO)** | $k = n$, mỗi lần để lại đúng 1 mẫu | Dữ liệu rất ít ($n < 50$). Ước lượng ít bias nhưng variance cao và tốn $n$ lần train |
| **Group K-Fold** | Các mẫu cùng "nhóm" (cùng bệnh nhân, cùng người dùng) không bị tách ra hai bên | Có dữ liệu lặp theo cá thể, nếu không sẽ rò rỉ nghiêm trọng |
| **TimeSeriesSplit** | Train luôn là quá khứ, validation là tương lai | Dữ liệu chuỗi thời gian, không được xáo trộn ngẫu nhiên |

Báo cáo đúng cách: không phải "accuracy = 0.94" mà là "accuracy = 0.94 ± 0.02 (5-fold CV)". Độ lệch chuẩn cho biết kết quả đáng tin đến đâu; hai model chênh nhau 0.005 mà độ lệch chuẩn 0.03 thì thực chất là ngang nhau.

### Nested CV: vì sao không được dùng test set để tune

Khi bạn chạy `GridSearchCV` trên 50 tổ hợp tham số rồi lấy tổ hợp tốt nhất, điểm CV của tổ hợp đó đã bị thiên lệch lạc quan: bạn đã chọn ra cái may mắn nhất trong 50 lần thử. Nếu tune trên chính tập test, con số báo cáo không còn là ước lượng hiệu năng nữa.

**Nested CV** tách bạch hai việc đó bằng hai vòng lặp lồng nhau:

```
Vòng ngoài (5 fold): dùng để ước lượng hiệu năng
   với mỗi fold ngoài:
        Vòng trong (5 fold, chỉ trên phần train của fold ngoài): dùng để chọn hyperparameter
        train lại với tham số tốt nhất trên toàn bộ phần train
        chấm điểm trên fold ngoài (dữ liệu chưa từng ảnh hưởng tới việc chọn tham số)
```

```python
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.svm import SVC
inner = StratifiedKFold(5, shuffle=True, random_state=0)
outer = StratifiedKFold(5, shuffle=True, random_state=1)
gs = GridSearchCV(SVC(), {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1]}, cv=inner)
scores = cross_val_score(gs, X, y, cv=outer)     # ước lượng không thiên lệch
print(f'{scores.mean():.3f} ± {scores.std():.3f}')
```

Ba tập, ba vai trò: train để học tham số; validation để chọn hyperparameter và ngưỡng; test chỉ được nhìn một lần duy nhất, ở cuối cùng, để báo cáo. Nhìn test nhiều lần thì test đã biến thành validation, và con số cuối cùng không còn là ước lượng khách quan.


## 14. Chỉ số cho bài toán hồi quy (bảng tra nhanh)

Bài này nói về phân loại, nhưng đề thi và đồ án thường hỏi cả hồi quy. Bảng dưới để tra nhanh (chi tiết xem lại Lab 01).

| Chỉ số | Công thức | Đơn vị | Đặc điểm |
|---|---|---|---|
| **MAE** | $\dfrac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | Cùng đơn vị $y$ | Dễ giải thích ("sai trung bình 12 triệu"). Bền với outlier |
| **MSE** | $\dfrac{1}{n}\sum (y_i - \hat{y}_i)^2$ | Đơn vị $y$ bình phương | Phạt nặng sai số lớn. Khó đọc vì sai đơn vị |
| **RMSE** | $\sqrt{\text{MSE}}$ | Cùng đơn vị $y$ | Phổ biến nhất. Luôn $\ge$ MAE; RMSE lớn hơn MAE nhiều là dấu hiệu có vài sai số rất lớn |
| **$R^2$** | $1 - \dfrac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$ | Không đơn vị | "Model giải thích được bao nhiêu % phương sai". $R^2 = 0$ nghĩa là tệ ngang việc luôn đoán $\bar{y}$; có thể âm trên tập test |
| **$R^2$ hiệu chỉnh** | $1-(1-R^2)\dfrac{n-1}{n-p-1}$ | Không đơn vị | Phạt việc thêm feature vô ích ($p$ = số feature). Dùng khi so sánh model khác số feature |
| **MAPE** | $\dfrac{100\%}{n}\sum\left\lvert\dfrac{y_i-\hat{y}_i}{y_i}\right\rvert$ | % | Dễ nói với sếp, nhưng có nhiều điểm cần lưu ý (xem dưới) |

### Ba điểm cần lưu ý với MAPE

1. Chia cho 0 hoặc gần 0. Nếu $y_i = 0$ thì MAPE = $\infty$; nếu $y_i = 0.01$ thì một sai số 0.05 tạo ra 500% và làm hỏng cả trung bình. Không dùng MAPE khi $y$ có thể gần 0 (nhiệt độ theo °C, lợi nhuận, số đơn hàng thưa...).
2. Bất đối xứng. Đoán thừa bị phạt tối đa vô hạn, còn đoán thiếu bị phạt tối đa 100% (khi $\hat{y}=0$). Vì thế tối thiểu hoá MAPE khiến model có thiên hướng đoán thấp một cách hệ thống.
3. Không so sánh được giữa các tập dữ liệu có thang giá trị khác nhau, dù trông có vẻ đã chuẩn hoá.

Các lựa chọn thay thế: sMAPE (đối xứng hơn), MASE (chia cho sai số của baseline naive, hợp với chuỗi thời gian), hoặc MAE trên log(y) khi $y$ dương và trải nhiều bậc.


## 15. Bảng chốt: bài toán nào thì nhìn chỉ số nào

| Bài toán | Sai lầm nào tốn kém nhất | Chỉ số chính | Kèm theo | Ghi chú về ngưỡng |
|---|---|---|---|---|
| **Lọc spam** | FP (mail thật vào thùng rác) | Precision, $F_{0.5}$ | FPR, confusion matrix | Ngưỡng cao (khoảng 0.9 trở lên) |
| **Sàng lọc ung thư** | FN (bỏ sót bệnh) | Recall / Sensitivity, $F_2$ | Specificity, NPV | Ngưỡng rất thấp theo $C_{FP}/(C_{FP}+C_{FN})$ |
| **Phát hiện gian lận thẻ** | Mất cân bằng nặng (~0.1% dương) | Average Precision (PR-AUC), MCC | Precision@k (k giao dịch nhân viên kịp kiểm tra) | Chọn theo năng lực điều tra, không theo 0.5 |
| **Xếp hạng / gợi ý** | Thứ tự, không phải nhãn | ROC-AUC, NDCG, MAP@k | Precision@k, Recall@k | Không cần ngưỡng, chỉ cần thứ tự |
| **Chấm điểm tín dụng** | Cần xác suất để tính vốn | Brier score, log loss, calibration curve | ROC-AUC (Gini $= 2\text{AUC}-1$) | Ngưỡng theo ma trận chi phí |
| **Phân loại ảnh nhiều lớp cân bằng** | Mọi lớp như nhau | Accuracy, macro-F1 | Confusion matrix đầy đủ | Argmax, không có ngưỡng |
| **Phân loại nhiều lớp mất cân bằng** | Lớp hiếm bị bỏ rơi | macro-F1, balanced accuracy | `classification_report` từng lớp | Có thể tinh chỉnh ngưỡng từng lớp |
| **Chẩn đoán y khoa hỗ trợ bác sĩ** | Niềm tin của người dùng | Sensitivity và Specificity (báo cáo cặp) | PPV/NPV theo tỷ lệ mắc thực tế | Bàn với chuyên gia lâm sàng |
| **Kiểm soát chất lượng sản xuất** | FN (hàng lỗi lọt ra) | Recall, $F_2$ | Chi phí kiểm tra thêm | Theo ma trận chi phí |

### Ba câu hỏi cần trả lời trước khi chọn chỉ số

1. Lớp dương chiếm bao nhiêu phần trăm? Dưới khoảng 10% thì bỏ accuracy và ROC-AUC làm chỉ số chính, chuyển sang AP, MCC hoặc macro-F1.
2. FP và FN, cái nào tốn kém hơn và hơn bao nhiêu lần? Con số đó cho $\beta$ của $F_\beta$ và ngưỡng $t^*$.
3. Đầu ra được dùng làm gì: nhãn, thứ tự, hay xác suất? Nhãn thì dùng F-score; thứ tự thì dùng AUC hoặc NDCG; xác suất thì dùng Brier và calibration.

Nguyên tắc chung: báo cáo ít nhất hai chỉ số cùng với confusion matrix, và kèm baseline. Một con số duy nhất luôn bỏ sót điều gì đó.


## Tổng kết

1. **Confusion matrix** là gốc, mọi chỉ số đều suy ra từ TP/TN/FP/FN.
2. Khi mất cân bằng, accuracy gây hiểu nhầm; dùng precision, recall, F1.
3. **Precision và Recall** là cặp đối kháng, chọn theo bài toán:
   - Spam filter: precision quan trọng.
   - Cancer screening: recall quan trọng.
4. F1 cân bằng hai chỉ số, AUC đánh giá trên toàn bộ ngưỡng.
5. **Multiclass**: macro / weighted / micro, biết khi nào dùng cái nào.

# BÀI TẬP VỀ NHÀ

## Bài 1: Wine dataset
Dùng `from sklearn.datasets import load_wine`. Train một model bất kỳ (KNN, RF, hay SVM). Báo cáo:
1. Confusion matrix.
2. Classification report.
3. F1 macro so với weighted: khác nhau thế nào?
4. Lớp nào model làm tệ nhất? Tại sao?

## Bài 2: Threshold tuning
Train logistic regression trên Breast Cancer. Quét ngưỡng từ 0.1 đến 0.9 (bước 0.05). Vẽ Precision và Recall theo ngưỡng. Tại ngưỡng nào F1 đạt max?

*Gợi ý:* `proba > threshold` thay vì gọi `predict()`.

## Bài 3: Precision-Recall curve
PR curve thường nói nhiều hơn ROC khi class mất cân bằng. Vẽ PR curve cho Breast Cancer (dùng lại `y_test`, `y_proba` của Thực hành 3; nếu đã chạy phần Iris thì tính lại hai biến này):
```python
from sklearn.metrics import precision_recall_curve, average_precision_score
p, r, _ = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
```
Vẽ kèm tiêu đề ghi AP. So với AUC ROC.

## Bài 4: Cohen's Kappa và MCC
Tìm hiểu hai chỉ số nâng cao:
- **Cohen's Kappa**: so accuracy với accuracy ngẫu nhiên.
- **MCC** (Matthews Correlation Coefficient): được xem là chỉ số cân bằng nhất cho bài nhị phân mất cân bằng.

Tính cả hai trên Breast Cancer (dùng KNN). So sánh với F1.

*Gợi ý:* `from sklearn.metrics import cohen_kappa_score, matthews_corrcoef`.

## Bài 5: Class imbalance handling
Sinh dữ liệu mất cân bằng 90/10 (1000 mẫu, 2 feature). Train KNN. So sánh:
1. Default (không xử lý imbalance).
2. Oversampling lớp ít với SMOTE: `pip install imbalanced-learn`, `from imblearn.over_sampling import SMOTE`.
3. Class weight: `KNeighborsClassifier` không có, nhưng `LogisticRegression(class_weight='balanced')` có.

Báo cáo F1 cho lớp ít trong từng trường hợp.